# **1. Import Library dan Load Dataset Kotor**
> Cell ini adalah titik awal seluruh proses cleansing. Tujuannya menyiapkan semua kebutuhan sebelum data diolah: mengimpor library, menghubungkan Colab ke Google Drive, membaca file Excel, dan menampilkan data awal sebagai baseline. Library pandas dipakai untuk semua operasi DataFrame, re (regex) dipakai di Step 8 untuk membersihkan format angka, dan drive dari google.colab dipakai untuk mengakses folder Drive. Setelah Drive di-mount, path file disimpan di variabel file_path, lalu file dibaca dengan pd.read_excel() dan hasilnya disimpan di variabel df. Terakhir, seluruh isi dataset ditampilkan memakai to_string(index=False) supaya nomor indeks pandas tidak ikut tercetak.


In [ ]:
# Import library pandas untuk mengolah data tabular (DataFrame)
import pandas as pd

# Import library re untuk operasi regex di step pembersihan angka
import re

# Import modul drive dari google.colab untuk akses Google Drive
from google.colab import drive

# Mount Google Drive ke Colab supaya file di Drive bisa diakses
drive.mount('/content/drive')

# Simpan path file dataset ke variabel file_path
file_path = "/content/drive/MyDrive/Datmin2418054/Dataset Anak Putus Sekolah Karena Ekonomi Di Indonesia.xlsx"

# Baca file Excel dari Drive, simpan ke DataFrame df
df = pd.read_excel(file_path)

# Tampilkan seluruh isi dataset awal tanpa nomor indeks
print("DATA AWAL:")
print(df.to_string(index=False))

# Tampilkan info ringkas dataset awal
print("\nINFO DATASET AWAL:")
print(f"Jumlah baris : {len(df)}")          # Hitung jumlah baris
print(f"Jumlah kolom : {len(df.columns)}")  # Hitung jumlah kolom
print(f"Kolom        : {df.columns.tolist()}")  # Daftar nama kolom

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATA AWAL:
 id   nama  usia jenis_kelamin jenjang  kelas            provinsi kabupaten_kota pekerjaan_orang_tua penghasilan_orang_tua  jumlah_tanggungan alasan_putus tanggal_putus status_sekarang
  1   Andi    12             L      SD      6          Jawa Barat        Bandung               Buruh               1200000                  4      ekonomi    2023-06-01   Tidak bekerja
  2   Budi    13     perempuan     smp      1               jabar        bandung               buruh                 1.2jt                  4      Ekonomi    01/07/2023   Tidak bekerja
  3  Citra    14             L     SMP      2         Jawa Tengah       Semarang              Petani          Rp 1.500.000                  3          NaN    2022-06-15   Tidak bekerja
  4   Dedi    10             P      SD      4              Jateng       Semarang              petani          Rp 1.500.0

# **2. Menangani Missing Value dan Memperbaiki isi Kolom Tertentu**

> Cell ini mengisi sel kosong pada kolom jenis_kelamin dan alasan_putus. Pertama, program menghitung jumlah sel kosong dengan df.isnull().sum() dan menyimpannya sebagai pembanding. Lalu program menyaring baris yang jenis_kelamin-nya kosong (ditemukan Budi, id=2). Setelah itu, program mengisi jenis_kelamin berdasarkan kamus mapping_nama yang memetakan nama ke jenis kelamin (misal "Budi": "L"), dan mengisi alasan_putus yang kosong dengan "Ekonomi". Program lalu menghitung ulang sel kosong, menampilkan tabel perbandingan sebelum vs sesudah, dan memastikan baris id 2, 7, 9, 16 sudah terisi dengan benar.





In [ ]:
print("="*60)
print("STEP 2: MISSING VALUE")
print("="*60)

# Langkah 1: hitung sel kosong sebelum cleansing
missing_sebelum = df.isnull().sum()

print("\nSEBELUM CLEANSING:")
tabel_sebelum = missing_sebelum.reset_index()
tabel_sebelum.columns = ["Kolom", "Jumlah Missing"]
print(tabel_sebelum.to_string(index=False))

# Langkah 2: cek baris mana yang jenis kelaminnya kosong
print("\nBARIS DENGAN JENIS_KELAMIN KOSONG:")
print(df[df["jenis_kelamin"].isnull()][["id", "nama"]].to_string(index=False))

# Langkah 3: kamus nama -> jenis kelamin
mapping_nama = {
    "Budi": "L",
    "Gilang": "L",
    "Irfan": "L",
    "Putri": "P",
}

# Langkah 4: isi jenis_kelamin berdasarkan nama
for nama, jk in mapping_nama.items():
    df.loc[df["nama"] == nama, "jenis_kelamin"] = jk

# Langkah 5: isi alasan_putus yang kosong dengan "Ekonomi"
df["alasan_putus"] = df["alasan_putus"].fillna("Ekonomi")

# Langkah 6: hitung sel kosong sesudah cleansing
missing_sesudah = df.isnull().sum()

print("\nSESUDAH CLEANSING:")
tabel_sesudah = missing_sesudah.reset_index()
tabel_sesudah.columns = ["Kolom", "Jumlah Missing"]
print(tabel_sesudah.to_string(index=False))

# Langkah 7: tampilkan tabel perbandingan
print("\nTABEL PERBANDINGAN:")
perbandingan = pd.DataFrame({
    "Kolom": df.columns,
    "Sebelum": missing_sebelum.values,
    "Sesudah": missing_sesudah.values
})
print(perbandingan.to_string(index=False))

# Langkah 8: cek hasil pengisian
print("\nHASIL PENGISIAN JENIS KELAMIN:")
print(df[df["id"].isin([2, 7, 9, 16])][["id", "nama", "jenis_kelamin"]].to_string(index=False))

STEP 2: MISSING VALUE

SEBELUM CLEANSING:
                Kolom  Jumlah Missing
                   id               0
                 nama               0
                 usia               0
        jenis_kelamin               3
              jenjang               0
                kelas               0
             provinsi               0
       kabupaten_kota               0
  pekerjaan_orang_tua               0
penghasilan_orang_tua               0
    jumlah_tanggungan               0
         alasan_putus               4
        tanggal_putus               0
      status_sekarang               0

BARIS DENGAN JENIS_KELAMIN KOSONG:
 id   nama
  7 Gilang
  9  Irfan
 16  Putri

SESUDAH CLEANSING:
                Kolom  Jumlah Missing
                   id               0
                 nama               0
                 usia               0
        jenis_kelamin               0
              jenjang               0
                kelas               0
             provinsi 

# **3. Menghapus Data Duplikat agar Tidak Ada Redundansi**

> Cell ini mendeteksi dan menghapus baris yang isinya sama persis. Program menghitung duplikat dengan membuang kolom id dulu (karena id unik per baris, kalau ikut dicek tidak akan ada duplikat yang terdeteksi), lalu menyimpan jumlah baris sebelum penghapusan. Setelah dijalankan, ditemukan 1 duplikat (id=50 identik dengan id=1), sehingga program menghapusnya dengan drop_duplicates(subset=df.columns.drop("id")) dan jumlah baris turun dari 50 menjadi 49. Terakhir, program menghitung ulang duplikat (harusnya 0) dan menampilkan tabel perbandingan.





In [ ]:
print("="*60)
print("STEP 3: DUPLIKAT")
print("="*60)

jml_sebelum = df.drop(columns=["id"]).duplicated().sum()
baris_sebelum = len(df)
print(f"\nSEBELUM: Jumlah duplikat = {jml_sebelum}")
print(f"SEBELUM: Jumlah baris    = {baris_sebelum}")

df = df.drop_duplicates(subset=df.columns.drop("id"))

jml_sesudah = df.drop(columns=["id"]).duplicated().sum()
baris_sesudah = len(df)
print(f"\nSESUDAH: Jumlah duplikat = {jml_sesudah}")
print(f"SESUDAH: Jumlah baris    = {baris_sesudah}")

tabel = pd.DataFrame({
    "Keterangan": ["Jumlah Duplikat", "Jumlah Baris"],
    "Sebelum": [jml_sebelum, baris_sebelum],
    "Sesudah": [jml_sesudah, baris_sesudah]
})
print("\nTABEL PERBANDINGAN:")
print(tabel.to_string(index=False))

STEP 3: DUPLIKAT

SEBELUM: Jumlah duplikat = 1
SEBELUM: Jumlah baris    = 50

SESUDAH: Jumlah duplikat = 0
SESUDAH: Jumlah baris    = 49

TABEL PERBANDINGAN:
     Keterangan  Sebelum  Sesudah
Jumlah Duplikat        1        0
   Jumlah Baris       50       49



# **4. Mengubah Format Tanggal Menjadi Datetime agar Sesuai untuk Analisis Berbasis Waktu**

> Cell ini mengubah tipe data kolom tanggal_putus dari teks (string) menjadi datetime, supaya bisa dipakai untuk analisis berbasis waktu. Program menampilkan tipe data dan isi kolom sebelum konversi, lalu memakai pd.to_datetime() dengan format="mixed" dan dayfirst=True supaya bisa membaca format tanggal yang campur (ada 2023-06-01, ada 01/07/2023). Hasilnya, tipe data berubah jadi datetime64[ns], tidak ada yang gagal konversi, dan program menampilkan tabel perbandingan



In [ ]:
print("="*60)
print("STEP 4: FORMAT TANGGAL")
print("="*60)

print("\nSEBELUM CLEANSING:")
print(f"Tipe data    : {df['tanggal_putus'].dtype}")
print(df[["tanggal_putus"]].to_string(index=False))

df["tanggal_putus"] = pd.to_datetime(
    df["tanggal_putus"], errors="coerce", format="mixed", dayfirst=True
)

print("\nSESUDAH CLEANSING:")
print(f"Tipe data    : {df['tanggal_putus'].dtype}")
print(f"Gagal konversi: {df['tanggal_putus'].isnull().sum()}")
print(df[["tanggal_putus"]].to_string(index=False))

tabel = pd.DataFrame({
    "Keterangan": ["Tipe Data", "Gagal Konversi"],
    "Sebelum": ["object", "-"],
    "Sesudah": [str(df["tanggal_putus"].dtype), df["tanggal_putus"].isnull().sum()]
})
print("\nTABEL PERBANDINGAN:")
print(tabel.to_string(index=False))

STEP 4: FORMAT TANGGAL

SEBELUM CLEANSING:
Tipe data    : object
tanggal_putus
   2023-06-01
   01/07/2023
   2022-06-15
   15/06/2022
   2023-06-20
   20/06/2024
   2022-05-10
   10/05/2023
   2023-07-01
   01/07/2024
   2022-08-15
   15/08/2023
   2023-09-01
   01/09/2024
   2022-03-20
   20/03/2023
   2023-04-10
   10/04/2024
   2022-11-05
   05/11/2023
   2022-12-01
   01/12/2023
   2023-01-15
   15/01/2024
   2022-02-20
   20/02/2023
   2022-03-10
   10/03/2023
   2023-04-25
   25/04/2024
   2022-05-30
   30/05/2023
   2023-06-05
   05/06/2022
   2023-07-10
   10/07/2024
   2023-08-15
   15/08/2024
   2022-09-20
   20/09/2023
   2023-10-25
   25/10/2024
   2022-11-30
   30/11/2023
   2023-12-05
   05/12/2024
   2023-01-10
   10/01/2024
   2023-06-01

SESUDAH CLEANSING:
Tipe data    : datetime64[ns]
Gagal konversi: 0
tanggal_putus
   2023-06-01
   2023-07-01
   2022-06-15
   2022-06-15
   2023-06-20
   2024-06-20
   2022-05-10
   2023-05-10
   2023-07-01
   2024-07-01
   2022-08-15

# **5. Menormalisasi Data Teks agar Lebih Konsisten**

> Cell ini menyeragamkan penulisan teks pada kolom kategorikal supaya konsisten. Program menampilkan kolom provinsi, kabupaten_kota, jenjang, jenis_kelamin, alasan_putus, dan status_sekarang sebelum normalisasi, lalu melakukan tiga hal: strip spasi dengan .str.strip(), atur kapitalisasi dengan .str.title() (provinsi/kabupaten/alasan/status) dan .str.upper() (jenjang), serta mengganti singkatan provinsi seperti jabar → Jawa Barat dan jatim → Jawa Timur memakai kamus mapping_prov. Hasilnya ditampilkan untuk memastikan semuanya konsisten.



In [ ]:
print("="*60)
print("STEP 5: NORMALISASI TEKS")
print("="*60)

kolom_teks = ["provinsi", "kabupaten_kota", "jenjang", "jenis_kelamin", "alasan_putus", "status_sekarang"]

print("\nSEBELUM CLEANSING:")
print(df[kolom_teks].to_string(index=False))

df["provinsi"] = df["provinsi"].str.strip().str.title()
df["kabupaten_kota"] = df["kabupaten_kota"].str.strip().str.title()
df["alasan_putus"] = df["alasan_putus"].str.strip().str.title()
df["status_sekarang"] = df["status_sekarang"].str.strip().str.title()
df["jenjang"] = df["jenjang"].str.strip().str.upper()
df["jenis_kelamin"] = df["jenis_kelamin"].str.strip()

mapping_prov = {
    "Jabar": "Jawa Barat", "Jateng": "Jawa Tengah", "Jatim": "Jawa Timur",
    "Sumut": "Sumatera Utara", "Sulsel": "Sulawesi Selatan",
    "Sulut": "Sulawesi Utara", "Sulteng": "Sulawesi Tengah",
    "Ntt": "Nusa Tenggara Timur", "Ntb": "Nusa Tenggara Barat",
    "Kalbar": "Kalimantan Barat", "Kalsel": "Kalimantan Selatan",
    "Kaltim": "Kalimantan Timur", "Dki Jakarta": "DKI Jakarta",
    "Diy": "DI Yogyakarta"
}
df["provinsi"] = df["provinsi"].replace(mapping_prov)

print("\nSESUDAH CLEANSING:")
print(df[kolom_teks].to_string(index=False))

STEP 5: NORMALISASI TEKS

SEBELUM CLEANSING:
           provinsi kabupaten_kota jenjang jenis_kelamin alasan_putus status_sekarang
         Jawa Barat        Bandung      SD             L      ekonomi   Tidak bekerja
              jabar        bandung     smp             L      Ekonomi   Tidak bekerja
        Jawa Tengah       Semarang     SMP             L      Ekonomi   Tidak bekerja
             Jateng       Semarang      SD             P      Ekonomi   Tidak bekerja
         Jawa Timur       Surabaya     SMA     Laki-laki      Ekonomi         Bekerja
              Jatim       Surabaya     sma             P      ekonomi         Bekerja
     Sumatera Utara          Medan      SD             L      ekonomi   Tidak bekerja
              Sumut          Medan     SMP             P      Ekonomi   Tidak bekerja
   Sulawesi Selatan       Makassar     SMP             L      ekonomi   Tidak bekerja
             Sulsel       Makassar     SMA             P      ekonomi         Bekerja
Nusa Teng

# **6. Memastikan Data Numerik Tidak Mengandung Nilai Negatif**

> Cell ini memastikan tidak ada nilai negatif pada kolom usia, kelas, dan jumlah_tanggungan. Program mengecek tiap kolom dengan kondisi < 0, menampilkan hasilnya dalam tabel, lalu mengganti nilai negatif jadi 0 memakai max(x, 0) (kalau tidak ada, tidak ada yang berubah). Setelah itu, program cek ulang untuk memastikan semuanya sudah non-negatif, dan hasilnya ditampilkan dalam tabel perbandingan.


In [ ]:
print("="*60)
print("STEP 6: NILAI NEGATIF")
print("="*60)

numeric_cols = ["usia", "kelas", "jumlah_tanggungan"]

print("\nSEBELUM CLEANSING:")
sebelum = []
for col in numeric_cols:
    sebelum.append({"Kolom": col, "Ada Negatif": (df[col] < 0).any()})
print(pd.DataFrame(sebelum).to_string(index=False))

for col in numeric_cols:
    df[col] = df[col].apply(lambda x: max(x, 0) if pd.notnull(x) else x)

print("\nSESUDAH CLEANSING:")
sesudah = []
for col in numeric_cols:
    sesudah.append({"Kolom": col, "Ada Negatif": (df[col] < 0).any()})
print(pd.DataFrame(sesudah).to_string(index=False))

STEP 6: NILAI NEGATIF

SEBELUM CLEANSING:
            Kolom  Ada Negatif
             usia        False
            kelas        False
jumlah_tanggungan        False

SESUDAH CLEANSING:
            Kolom  Ada Negatif
             usia        False
            kelas        False
jumlah_tanggungan        False


# **7. Standardisasi Kategori Jenis Kelamin agar Konsisten**

> Cell ini menyeragamkan nilai kolom jenis_kelamin jadi cuma L (laki-laki) atau P (perempuan). Program menampilkan nilai sebelum, lalu memetakan semua variasi seperti l, Laki-laki, perempuan, p ke nilai baku pakai kamus mapping.





In [ ]:
print("="*60)
print("STEP 7: STANDARDISASI JENIS KELAMIN")
print("="*60)

print("\nSEBELUM CLEANSING:")
print(df[["id", "jenis_kelamin"]].to_string(index=False))
print(f"\nNilai unik sebelum: {df['jenis_kelamin'].unique()}")

mapping = {
    "L": "L", "l": "L", "Laki-laki": "L", "Laki-Laki": "L", "laki-laki": "L",
    "P": "P", "p": "P", "Perempuan": "P", "perempuan": "P",
    "Tidak Diketahui": "Tidak Diketahui"
}
df["jenis_kelamin"] = df["jenis_kelamin"].str.strip().map(mapping)

print("\nSESUDAH CLEANSING:")
print(df[["id", "jenis_kelamin"]].to_string(index=False))
print(f"\nNilai unik sesudah: {df['jenis_kelamin'].unique()}")
print(f"Jumlah kosong: {df['jenis_kelamin'].isnull().sum()}")

STEP 7: STANDARDISASI JENIS KELAMIN

SEBELUM CLEANSING:
 id jenis_kelamin
  1             L
  2             L
  3             L
  4             P
  5     Laki-laki
  6             P
  7             L
  8             P
  9             L
 10             P
 11             L
 12             P
 13             L
 14             P
 15             L
 16             P
 17             L
 18             P
 19             L
 20             P
 21             L
 22             P
 23             L
 24             P
 25             L
 26             P
 27             L
 28             P
 29             L
 30             P
 31             L
 32             P
 33             L
 34             P
 35             L
 36             P
 37             L
 38             P
 39             L
 40             P
 41             L
 42             P
 43             L
 44             P
 45             L
 46             P
 47             L
 48             P
 49             L

Nilai unik sebelum: ['L' 'P' 'Laki-laki']



# **8. Menyeragamkan Format Angka dan Mata Uang**

> Cell ini menyeragamkan kolom penghasilan_orang_tua yang formatnya masih bercampur (ada 1200000, 1.2jt, Rp 1.500.000, 900rb, 2jt). Program mendefinisikan fungsi bersihkan_angka() yang mendeteksi satuan jt (juta) dan rb (ribu), menghapus simbol Rp, titik, dan koma, lalu mengubah semua jadi integer. Fungsi ini diterapkan ke seluruh kolom, dan hasilnya ditampilkan bersama tipe data baru yang sudah int64.



In [ ]:
print("="*60)
print("STEP 8: FORMAT ANGKA & MATA UANG")
print("="*60)

print("\nSEBELUM CLEANSING:")
print(df["penghasilan_orang_tua"].to_string(index=False))
print(f"\nTipe data: {df['penghasilan_orang_tua'].dtype}")

def bersihkan_angka(x):
    if pd.isnull(x):
        return 0
    if isinstance(x, (int, float)):
        return int(x)
    x = str(x).lower().replace("rp", "").strip()
    if "jt" in x:
        return int(float(x.replace("jt", "").replace(",", ".")) * 1_000_000)
    if "rb" in x:
        return int(float(re.sub(r"[^0-9.,]", "", x).replace(",", ".")) * 1_000)
    x = x.replace(".", "").replace(",", "").strip()
    x = re.sub(r"[^0-9]", "", x)
    return int(x) if x else 0

df["penghasilan_orang_tua"] = df["penghasilan_orang_tua"].apply(bersihkan_angka).astype(int)

print("\nSESUDAH CLEANSING:")
print(df["penghasilan_orang_tua"].to_string(index=False))
print(f"\nTipe data: {df['penghasilan_orang_tua'].dtype}")

STEP 8: FORMAT ANGKA & MATA UANG

SEBELUM CLEANSING:
     1200000
       1.2jt
Rp 1.500.000
Rp 1.500.001
     2000000
         2jt
      900000
       900rb
Rp 1.000.000
         1jt
      700000
       700rb
     1800000
       1.8jt
      800000
       800rb
     2500000
       2.5jt
     1500000
       1.5jt
     1000000
         1jt
Rp 1.200.000
       1.2jt
      900000
       900rb
      700000
       700rb
     1500000
       1.5jt
     1800000
       1.8jt
     1200000
       1.2jt
      800000
       800rb
      900000
       900rb
     1000000
         1jt
     1100000
       1.1jt
      700000
       700rb
     1600000
       1.6jt
     2200000
       2.2jt
     1200000

Tipe data: object

SESUDAH CLEANSING:
1200000
1200000
1500000
1500001
2000000
2000000
 900000
 900000
1000000
1000000
 700000
 700000
1800000
1800000
 800000
 800000
2500000
2500000
1500000
1500000
1000000
1000000
1200000
1200000
 900000
 900000
 700000
 700000
1500000
1500000
1800000
1800000
1200000
1200000

# **9. Menangani Outlier agar Nilai Ekstrem Tidak Mengganggu Analisis**

> Cell ini menangani outlier pada kolom usia. Program menampilkan statistik deskriptif sebelum, menghitung batas wajar pakai metode IQR (Q1 - 1.5 * IQR dan Q3 + 1.5 * IQR), lalu memfilter data agar hanya usia 6–18 tahun (rentang wajar anak sekolah). Program juga menampilkan distribusi frekuensi usia dan tabel perbandingan sebelum vs sesudah.



In [ ]:
print("="*60)
print("STEP 9: OUTLIER (USIA)")
print("="*60)

print("\nSEBELUM CLEANSING:")
print(df["usia"].describe().to_string())
print(f"Jumlah baris: {len(df)}")

Q1 = df["usia"].quantile(0.25)
Q3 = df["usia"].quantile(0.75)
IQR = Q3 - Q1
batas_bawah = Q1 - 1.5 * IQR
batas_atas = Q3 + 1.5 * IQR

print(f"\nBatas wajar IQR:")
print(f"  Q1          : {Q1}")
print(f"  Q3          : {Q3}")
print(f"  IQR         : {IQR}")
print(f"  Batas bawah : {batas_bawah}")
print(f"  Batas atas  : {batas_atas}")

outlier_iqr = df[(df["usia"] < batas_bawah) | (df["usia"] > batas_atas)]
print(f"\nJumlah outlier (IQR) : {len(outlier_iqr)}")

print("\nDistribusi usia:")
print(df["usia"].value_counts().sort_index().to_string())

df = df[(df["usia"] >= 6) & (df["usia"] <= 18)]

print("\nSESUDAH CLEANSING:")
print(df["usia"].describe().to_string())
print(f"Jumlah baris sesudah : {len(df)}")

tabel = pd.DataFrame({
    "Keterangan": ["Jumlah Baris", "Usia Min", "Usia Max", "Rata-rata", "Std Deviasi"],
    "Sebelum": [49, 10, 16, round(df["usia"].mean(), 2), round(df["usia"].std(), 2)],
    "Sesudah": [len(df), df["usia"].min(), df["usia"].max(), round(df["usia"].mean(), 2), round(df["usia"].std(), 2)]
})
print("\nTABEL PERBANDINGAN:")
print(tabel.to_string(index=False))

STEP 9: OUTLIER (USIA)

SEBELUM CLEANSING:
count    49.000000
mean     13.428571
std       1.791182
min      10.000000
25%      12.000000
50%      14.000000
75%      15.000000
max      16.000000
Jumlah baris: 49

Batas wajar IQR:
  Q1          : 12.0
  Q3          : 15.0
  IQR         : 3.0
  Batas bawah : 7.5
  Batas atas  : 19.5

Jumlah outlier (IQR) : 0

Distribusi usia:
usia
10    3
11    5
12    8
13    8
14    9
15    9
16    7

SESUDAH CLEANSING:
count    49.000000
mean     13.428571
std       1.791182
min      10.000000
25%      12.000000
50%      14.000000
75%      15.000000
max      16.000000
Jumlah baris sesudah : 49

TABEL PERBANDINGAN:
  Keterangan  Sebelum  Sesudah
Jumlah Baris    49.00    49.00
    Usia Min    10.00    10.00
    Usia Max    16.00    16.00
   Rata-rata    13.43    13.43
 Std Deviasi     1.79     1.79


# **10. Menghapus Whitespace dan Karakter Khusus agar Data Lebih Bersih**

> Cell ini membersihkan spasi berlebih dan karakter khusus pada semua kolom teks. Program menampilkan kondisi awal, lalu melakukan iterasi tiap kolom bertipe object dan menjalankan .str.strip(), mengganti karakter aneh seperti ??, N/A, -, dan string kosong jadi NA, serta menormalkan pekerjaan_orang_tua ke title case. Hasilnya ditampilkan bersama tabel perbandingan dan distribusi pekerjaan orang tua.



In [ ]:
print("="*60)
print("STEP 10: WHITESPACE & KARAKTER KHUSUS")
print("="*60)

print("\nSEBELUM CLEANSING:")
print(f"Jumlah missing       : {df.isnull().sum().sum()}")
print(f"Nilai unik pekerjaan : {df['pekerjaan_orang_tua'].unique()}")

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

df = df.replace(["??", "N/A", "-", ""], pd.NA)
df["pekerjaan_orang_tua"] = df["pekerjaan_orang_tua"].str.title()

print("\nSESUDAH CLEANSING:")
print(f"Jumlah missing       : {df.isnull().sum().sum()}")
print(f"Nilai unik pekerjaan : {df['pekerjaan_orang_tua'].unique()}")

tabel = pd.DataFrame({
    "Keterangan": ["Jumlah Missing", "Whitespace", "Karakter Khusus", "Nilai Unik Pekerjaan"],
    "Sebelum": [0, "Ada", "Ada", "Buruh, buruh, BURUH"],
    "Sesudah": [df.isnull().sum().sum(), "Bersih", "Bersih", ", ".join(df["pekerjaan_orang_tua"].dropna().unique())]
})
print("\nTABEL PERBANDINGAN:")
print(tabel.to_string(index=False))

print("\nDistribusi pekerjaan orang tua:")
print(df["pekerjaan_orang_tua"].value_counts().to_string())

STEP 10: WHITESPACE & KARAKTER KHUSUS

SEBELUM CLEANSING:
Jumlah missing       : 0
Nilai unik pekerjaan : ['Buruh' 'buruh' 'Petani' 'petani' 'Pedagang' 'pedagang' 'Nelayan'
 'nelayan' 'Karyawan' 'karyawan']

SESUDAH CLEANSING:
Jumlah missing       : 0
Nilai unik pekerjaan : ['Buruh' 'Petani' 'Pedagang' 'Nelayan' 'Karyawan']

TABEL PERBANDINGAN:
          Keterangan             Sebelum                                    Sesudah
      Jumlah Missing                   0                                          0
          Whitespace                 Ada                                     Bersih
     Karakter Khusus                 Ada                                     Bersih
Nilai Unik Pekerjaan Buruh, buruh, BURUH Buruh, Petani, Pedagang, Nelayan, Karyawan

Distribusi pekerjaan orang tua:
pekerjaan_orang_tua
Buruh       15
Petani      14
Nelayan     10
Pedagang     8
Karyawan     2


# **11. Menampilkan Hasil Akhir Cleansing dari 49 Baris Data**

> Cell ini menampilkan hasil akhir setelah seluruh tahap cleansing selesai. Program menampilkan informasi dataset akhir berupa tipe data tiap kolom, jumlah nilai non-null, dan penggunaan memori. Selanjutnya, program menampilkan jumlah baris dan kolom akhir, lalu menampilkan seluruh isi dataset sebanyak 49 baris tanpa ada yang terpotong. Di akhir, dataset disimpan ke file Excel bernama Dataset Anak Putus Sekolah Karena Ekonomi Di Indonesia clean.xlsx di Google Drive.



In [ ]:
from openpyxl import load_workbook
from google.colab import files

print("="*60)
print("STEP 11: HASIL AKHIR CLEANSING")
print("="*60)

print("\nINFO DATASET AKHIR:")
print(df.info())

print("\nJUMLAH BARIS & KOLOM:")
print(f"Total baris : {len(df)}")
print(f"Total kolom : {len(df.columns)}")

print("\nSELURUH DATA:")
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
print(df.to_string(index=False))

output_path = "/content/drive/MyDrive/Datmin2418054/Dataset Anak Putus Sekolah Karena Ekonomi Di Indonesia clean.xlsx"
df.to_excel(output_path, index=False)
print(f"\nFile tersimpan di Drive : {output_path}")

wb = load_workbook(output_path)
ws = wb.active
kolom_tanggal = list(df.columns).index("tanggal_putus") + 1
for row in ws.iter_rows(min_row=2, min_col=kolom_tanggal, max_col=kolom_tanggal):
    for cell in row:
        cell.number_format = "YYYY-MM-DD"
wb.save(output_path)
print("Format tanggal sudah diatur.")

files.download(output_path)
print("File sedang diunduh ke komputer...")

STEP 11: HASIL AKHIR CLEANSING

INFO DATASET AKHIR:
<class 'pandas.core.frame.DataFrame'>
Index: 49 entries, 0 to 48
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   id                     49 non-null     int64         
 1   nama                   49 non-null     object        
 2   usia                   49 non-null     int64         
 3   jenis_kelamin          49 non-null     object        
 4   jenjang                49 non-null     object        
 5   kelas                  49 non-null     int64         
 6   provinsi               49 non-null     object        
 7   kabupaten_kota         49 non-null     object        
 8   pekerjaan_orang_tua    49 non-null     object        
 9   penghasilan_orang_tua  49 non-null     int64         
 10  jumlah_tanggungan      49 non-null     int64         
 11  alasan_putus           49 non-null     object        
 12  tanggal_putus      

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File sedang diunduh ke komputer...


# **12.Kesimpulan**

>Proses data cleansing pada dataset Anak Putus Sekolah berhasil
dilakukan melalui 10 tahap penanganan, yaitu:
1.  Menangani missing value
2.  Menghapus data duplikat
3.  Mengubah format tanggal ke datetime
4.  Menormalisasi data teks
5.  Memastikan data numerik tidak negatif
6.  Menghapus kolom tidak relevan
7.  Menangani outlier
8.  Menyeragamkan format angka & mata uang
9.  Menyeragamkan kategori (standardisasi label)
10. Menghapus whitespace & karakter khusus

Jumlah baris awal  : 50

Jumlah baris akhir : 49

Jumlah kolom akhir : 14

Missing value      : 0

Duplikat           : 0


